# Pyramid CNN (U-Net) — Behavior Cloning

Encoder–decoder spatial policy over the board (U-Net backbone + 1×1 policy head).

- Input `(14, 23, 23)` → backbone `(32, 23, 23)` → policy `(9, 23, 23)`
- Channels `0–7`: move `dir + 4*split` from cell `(r, c)`; channel `8`: pass
- Reproducible `game_id` split · epoch loss/acc · save `bc/checkpoints/pyr_cnn.pt`

## 1. Imports & config

**Colab data flow**
1. Mount Drive (`MyDrive/general_data`)
2. Copy **all** `bc_shard_*.npz` → `/content/bc/data` (once per runtime)
3. Index all shards once · re-run **split only** to try different train/val/test fracs


In [17]:
# Colab only — mount Drive. Data lives at MyDrive/general_data.
try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass

DRIVE_DATA = "/content/drive/MyDrive/general_data"
LOCAL_DATA = "/content/bc/data"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Copy ALL shards Drive → Colab local SSD (once per runtime).
# Re-run split / train freely after this; skip re-copy if local already complete.
import shutil
from pathlib import Path

DRIVE_DATA = Path(globals().get("DRIVE_DATA") or "/content/drive/MyDrive/general_data")
LOCAL_DATA = Path("/content/bc/data")

drive_shards = sorted(DRIVE_DATA.glob("bc_shard_*.npz")) if DRIVE_DATA.exists() else []
local_shards = sorted(LOCAL_DATA.glob("bc_shard_*.npz")) if LOCAL_DATA.exists() else []

print(f"drive: {DRIVE_DATA}  shards={len(drive_shards)}")
print(f"local: {LOCAL_DATA}  shards={len(local_shards)}")

if local_shards and (not drive_shards or len(local_shards) >= len(drive_shards)):
    print(f"skip copy — local already has {len(local_shards)} shard(s) (full dataset)")
elif not drive_shards:
    print("no Drive shards found — mount Drive / check DRIVE_DATA")
else:
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    print(f"copying ALL {len(drive_shards)} shards → {LOCAL_DATA} ...")
    for i, src in enumerate(drive_shards, 1):
        dst = LOCAL_DATA / src.name
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            status = "skip"
        else:
            shutil.copy2(src, dst)
            status = "copied"
        if i == 1 or i == len(drive_shards) or i % 10 == 0 or status == "copied":
            print(f"  [{i}/{len(drive_shards)}] {status}  {src.name}")
    local_shards = sorted(LOCAL_DATA.glob("bc_shard_*.npz"))
    assert len(local_shards) >= len(drive_shards), (
        f"incomplete copy: local={len(local_shards)} drive={len(drive_shards)}"
    )
    print(f"done. local has ALL {len(local_shards)} shards — safe to change splits without re-copy")


drive: /content/drive/MyDrive/general_data  shards=74
local: /content/bc/data  shards=74
skip copy — local already has 74 shard(s) (full dataset)


In [19]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

_ROOT = Path.cwd().resolve()
if not (_ROOT / "generals").exists() and (_ROOT.parent / "generals").exists():
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

LOCAL_DATA = Path(globals().get("LOCAL_DATA") or "/content/bc/data")
DRIVE_DATA = Path(globals().get("DRIVE_DATA") or "/content/drive/MyDrive/general_data")

# Prefer local SSD copy of ALL shards (from copy cell). Drive is fallback only.
DATA_DIRS = [
    LOCAL_DATA,
    DRIVE_DATA,
    Path("bc/data"),
    Path("data"),
    Path("bc/bc/data"),
]
DATA_DIRS = [d for d in DATA_DIRS if d is not None]
DATA_DIR = next(
    (d for d in DATA_DIRS if d.exists() and list(d.glob("bc_shard_*.npz"))),
    DATA_DIRS[0],
)

GRID_SIZE = 23
# Training / paper-aligned hyperparams
NENVS = 128
NSTEPS = 6000
BATCH_SIZE = 600
TRUNCATION = 2000
LR = 5e-5          # heads training
LR_FINETUNE = 2e-5  # fine-tuning (unused unless you switch)
GAMMA = 1.0
GRAD_CLIP_NORM = 0.25
MAX_EPOCHS = 20
PATIENCE = 5
SEED = 0

TRAIN_FRAC = 0.10
VAL_FRAC = 0.010
TEST_FRAC = 0.010

CKPT_DIR = Path("/content/drive/MyDrive/generals_ai/bc/checkpoints")
CKPT_NAME = "pyr_cnn.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_shards = len(list(DATA_DIR.glob("bc_shard_*.npz"))) if DATA_DIR.exists() else 0
print(f"device: {DEVICE}")
print(f"ckpt dir: {CKPT_DIR.resolve()}")
print(f"data dir: {DATA_DIR.resolve()}  exists={DATA_DIR.exists()}  shards={n_shards}")
print(f"using LOCAL SSD" if DATA_DIR.resolve() == LOCAL_DATA.resolve() else "WARNING: reading from Drive (slow) — run copy cell")
print(f"split: train={TRAIN_FRAC:.0%} val={VAL_FRAC:.0%} test={TEST_FRAC:.0%}  seed={SEED}")
print(f"train: batch={BATCH_SIZE} lr={LR} clip={GRAD_CLIP_NORM} epochs<={MAX_EPOCHS}  nenvs={NENVS} nsteps={NSTEPS} trunc={TRUNCATION} γ={GAMMA}")
if n_shards == 0:
    print(
        "NO SHARDS FOUND — run mount + copy cells (Drive MyDrive/general_data → /content/bc/data)."
    )


device: cuda
ckpt dir: /content/drive/MyDrive/generals_ai/bc/checkpoints
data dir: /content/bc/data  exists=True  shards=74
using LOCAL SSD
split: train=10% val=1% test=1%  seed=0
train: batch=600 lr=5e-05 clip=0.25 epochs<=20  nenvs=128 nsteps=6000 trunc=2000 γ=1.0


## 2. Index all shards (lazy — do not load full `obs` into RAM)

Indexes **every** local shard once (`game_id` only). Re-run §3 to change splits without re-copy or re-index.


In [20]:
REQUIRED_KEYS = ("obs", "actions", "game_id", "player", "turn", "aug")

shard_paths_all = sorted(DATA_DIR.glob("bc_shard_*.npz"))
assert shard_paths_all, f"no shards found under {DATA_DIR}"

shard_paths: list[Path] = []
skipped: list[str] = []

for path in shard_paths_all:
    try:
        with np.load(path, allow_pickle=False) as z:
            keys = set(z.files)
            if not set(REQUIRED_KEYS).issubset(keys):
                skipped.append(f"{path.name} (missing {sorted(set(REQUIRED_KEYS) - keys)})")
                continue
            _ = z["game_id"][:1]
        shard_paths.append(path)
    except Exception as e:
        skipped.append(f"{path.name} ({type(e).__name__}: {e})")

if skipped:
    print(f"skipped {len(skipped)} incomplete/corrupt shard(s):")
    for s in skipped:
        print(f"  - {s}")

assert shard_paths, "no valid shards left after filtering"

shard_game_ids: list[np.ndarray] = []
all_game_ids_list: list[np.ndarray] = []
sample_index: list[tuple[int, int]] = []

for si, path in enumerate(shard_paths):
    with np.load(path, allow_pickle=False) as z:
        gids = z["game_id"].astype(str)
    shard_game_ids.append(gids)
    for li in range(len(gids)):
        sample_index.append((si, li))
    all_game_ids_list.append(gids)

game_ids = np.concatenate(all_game_ids_list, axis=0)
unique_games = np.unique(game_ids)
print(f"shards: {len(shard_paths)} valid / {len(shard_paths_all)} total")
print(f"samples: {len(sample_index):,}")
print(f"unique games: {len(unique_games):,}")
print(f"first shard: {shard_paths[0].name}")
print(f"last shard:  {shard_paths[-1].name}")

shards: 74 valid / 74 total
samples: 9,655,528
unique games: 2,368
first shard: bc_shard_0000_n32_t1001.npz
last shard:  bc_shard_0073_n32_t1001.npz


## 3. Reproducible split by `game_id`

Change `TRAIN_FRAC` / `VAL_FRAC` / `TEST_FRAC` in config and **re-run only this section** — no need to re-copy or re-index.
Same `SEED` + same shards → same games for a given frac.


In [21]:
def split_by_game_id(
    game_ids: np.ndarray,
    train_frac: float,
    val_frac: float,
    test_frac: float,
    seed: int = SEED,
):
    """Deterministic game-level split. No sample leakage across splits."""
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-9 or (
        train_frac + val_frac + test_frac <= 1.0
    ), "fractions should sum to <= 1"

    unique = np.unique(game_ids)
    unique = np.sort(unique)
    rng = np.random.default_rng(seed)
    unique = rng.permutation(unique)

    n = len(unique)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    n_test = int(n * test_frac)

    train_ids = set(unique[:n_train].tolist())
    val_ids = set(unique[n_train : n_train + n_val].tolist())
    test_ids = set(unique[n_train + n_val : n_train + n_val + n_test].tolist())

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)

    train_mask = np.isin(game_ids, list(train_ids))
    val_mask = np.isin(game_ids, list(val_ids))
    test_mask = np.isin(game_ids, list(test_ids))

    print(
        f"games:   train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}  "
        f"unused={n - len(train_ids) - len(val_ids) - len(test_ids)}  seed={seed}"
    )
    print(
        f"samples: train={int(train_mask.sum()):,}  val={int(val_mask.sum()):,}  "
        f"test={int(test_mask.sum()):,}"
    )
    return train_mask, val_mask, test_mask, {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids,
        "seed": seed,
        "ordered_unique": unique,
    }


train_mask, val_mask, test_mask, split_ids = split_by_game_id(
    game_ids, TRAIN_FRAC, VAL_FRAC, TEST_FRAC, seed=SEED
)
assert split_ids["train"].isdisjoint(split_ids["val"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["val"].isdisjoint(split_ids["test"])
print("leakage check passed")

games:   train=236  val=23  test=23  unused=2086  seed=0
samples: train=985,336  val=120,544  test=103,016
leakage check passed


## 4. Lazy multi-shard dataset + U-Net CNN policy

In [22]:
N_POLICY = 9  # 4 dirs × 2 splits + pass
PASS_CH = 8


def action_to_flat_index(act, grid_size: int = GRID_SIZE) -> int:
    """Map [pass, row, col, dir, split] → class in flattened (9, H, W)."""
    to_pass, row, col, direction, split = (int(x) for x in act)
    if to_pass:
        return PASS_CH  # channel 8 at (0, 0)
    ch = int(direction) + 4 * int(split)
    return ch + N_POLICY * (int(col) + grid_size * int(row))


def flat_index_to_action(idx: int, grid_size: int = GRID_SIZE) -> tuple[int, int, int, int, int]:
    """Inverse → (pass, row, col, dir, split)."""
    ch = idx % N_POLICY
    cell = idx // N_POLICY
    row, col = divmod(cell, grid_size)
    if ch == PASS_CH:
        return (1, 0, 0, 0, 0)
    direction, split = ch % 4, ch // 4
    return (0, row, col, direction, split)


import gc
from concurrent.futures import ThreadPoolExecutor, as_completed

# ~4GB/shard decompressed; with ~25GB RAM, 4 shards ≈ 16GB leaves headroom for model/batch.
SHARD_WINDOW = 4


def _load_shard_arrays(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load one compressed .npz (runs in a worker thread)."""
    with np.load(path, allow_pickle=False) as z:
        obs = np.asarray(z["obs"])
        actions = np.asarray(z["actions"])
    return obs, actions


class ShardCache:
    """Hold one window of decompressed shards in RAM.

    Flow: preload(4 shards in parallel) → train/eval → clear → next window.
    Shards are savez_compressed (~4GB each once loaded). Share one cache across
    train/val/test; never keep windows from two splits at once.
    """

    def __init__(self, paths: list[Path], maxsize: int = SHARD_WINDOW):
        self.paths = paths
        self.maxsize = max(1, int(maxsize))
        self._cache: dict[int, tuple[np.ndarray, np.ndarray]] = {}

    def clear(self):
        self._cache.clear()
        gc.collect()

    def preload(self, shard_indices: list[int]):
        """Load these shards concurrently; drop anything else first to free RAM."""
        want = list(dict.fromkeys(int(si) for si in shard_indices))
        if len(want) > self.maxsize:
            raise ValueError(f"window has {len(want)} shards > maxsize={self.maxsize}")

        want_set = set(want)
        dropped = False
        for si in list(self._cache):
            if si not in want_set:
                del self._cache[si]
                dropped = True
        if dropped:
            gc.collect()

        missing = [si for si in want if si not in self._cache]
        if not missing:
            return

        names = [self.paths[si].name for si in missing]
        print(f"  [shard] parallel load ({len(missing)}): {names}", flush=True)

        # Threads: I/O + zlib decompress release the GIL enough to overlap well.
        # (Processes would need to pickle ~4GB arrays back — worse.)
        with ThreadPoolExecutor(max_workers=len(missing)) as pool:
            futures = {
                pool.submit(_load_shard_arrays, self.paths[si]): si for si in missing
            }
            for fut in as_completed(futures):
                si = futures[fut]
                obs, actions = fut.result()
                self._cache[si] = (obs, actions)
                print(
                    f"  [shard] ready {self.paths[si].name}  obs={obs.shape}  "
                    f"~{obs.nbytes / 1e9:.2f}GB  cached={len(self._cache)}/{self.maxsize}",
                    flush=True,
                )

    def get(self, shard_idx: int):
        if shard_idx not in self._cache:
            raise KeyError(
                f"shard {shard_idx} not in cache — call preload() for its window first"
            )
        return self._cache[shard_idx]


def iter_shard_windows(
    entries: list[tuple[int, int]],
    window_size: int = SHARD_WINDOW,
    seed: int = 0,
    epoch: int = 0,
    shuffle: bool = True,
):
    """Yield (shard_window, dataset_indices) covering all entries once."""
    by_shard: dict[int, list[int]] = {}
    for ds_idx, (si, _) in enumerate(entries):
        by_shard.setdefault(si, []).append(ds_idx)

    rng = np.random.default_rng(seed + epoch)
    shards = list(by_shard.keys())
    if shuffle:
        rng.shuffle(shards)

    for i in range(0, len(shards), window_size):
        window = shards[i : i + window_size]
        idxs: list[int] = []
        for si in window:
            part = by_shard[si].copy()
            if shuffle:
                rng.shuffle(part)
            idxs.extend(part)
        # Mix samples across the loaded window so batches use all 4 shards.
        if shuffle and len(window) > 1:
            rng.shuffle(idxs)
        yield window, idxs


class MultiShardBCDataset(Dataset):
    def __init__(
        self,
        shard_paths: list[Path],
        sample_index: list[tuple[int, int]],
        mask: np.ndarray,
        cache: ShardCache | None = None,
    ):
        self.paths = shard_paths
        self.cache = cache if cache is not None else ShardCache(shard_paths)
        idxs = np.nonzero(mask)[0]
        self.entries = [sample_index[i] for i in idxs]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, i: int):
        si, li = self.entries[i]
        obs_arr, act_arr = self.cache.get(si)
        obs = torch.from_numpy(np.asarray(obs_arr[li], dtype=np.float32))
        act = np.asarray(act_arr[li], dtype=np.int64)
        y = torch.tensor(action_to_flat_index(act), dtype=torch.int64)
        return obs, y


class ConvReLU(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(self.conv(x))


class PyramidBackbone(nn.Module):
    """U-Net encoder–decoder → (B, 32, H, W).

    Spatial path (H=W=23):
      stem → L1 (skip1) → pool 11 → L2 (skip2) → pool 5 → bottleneck
      → up 11 + skip2 → up 23 + skip1 → 32 features
    """

    def __init__(self, in_channels: int = 14):
        super().__init__()
        # Encoder
        self.stem = ConvReLU(in_channels, 32)
        self.enc1 = nn.Sequential(ConvReLU(32, 32), ConvReLU(32, 32))
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(ConvReLU(32, 64), ConvReLU(64, 64))
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(ConvReLU(64, 128), ConvReLU(128, 128))

        # Decoder
        self.up_reduce2 = ConvReLU(128, 64)
        self.dec2 = nn.Sequential(ConvReLU(128, 64), ConvReLU(64, 64))
        self.up_reduce1 = ConvReLU(64, 32)
        self.dec1 = nn.Sequential(ConvReLU(64, 32), ConvReLU(32, 32))

    def forward(self, x):
        # Encoder
        x = self.stem(x)
        skip1 = self.enc1(x)          # 32 × 23 × 23
        x = self.pool1(skip1)         # 32 × 11 × 11
        skip2 = self.enc2(x)          # 64 × 11 × 11
        x = self.pool2(skip2)         # 64 × 5 × 5
        x = self.bottleneck(x)        # 128 × 5 × 5

        # Decoder — upsample to skip spatial size (odd grids: 5→11, 11→23)
        x = F.interpolate(x, size=skip2.shape[-2:], mode="nearest")
        x = self.up_reduce2(x)        # 64 × 11 × 11
        x = torch.cat([x, skip2], dim=1)  # 128 × 11 × 11
        x = self.dec2(x)              # 64 × 11 × 11

        x = F.interpolate(x, size=skip1.shape[-2:], mode="nearest")
        x = self.up_reduce1(x)        # 32 × 23 × 23
        x = torch.cat([x, skip1], dim=1)  # 64 × 23 × 23
        x = self.dec1(x)              # 32 × 23 × 23
        return x


class GeneralsBCNet(nn.Module):
    """U-Net backbone + 1×1 policy head → (B, 9, H, W)."""

    def __init__(self, in_channels: int = 14):
        super().__init__()
        self.backbone = PyramidBackbone(in_channels)
        self.policy = nn.Conv2d(in_channels=32, out_channels=N_POLICY, kernel_size=1)

    def forward(self, x):
        return self.policy(self.backbone(x))


def policy_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    b = logits.size(0)
    return nn.functional.cross_entropy(logits.view(b, -1), y)


## 5. Train / eval helpers

In [23]:
def set_seed(seed: int = SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)


def run_shard_windows(
    model: nn.Module,
    dataset: MultiShardBCDataset,
    cache: ShardCache,
    *,
    train: bool,
    opt: torch.optim.Optimizer | None = None,
    epoch: int = 0,
    shuffle: bool = False,
    window_size: int = SHARD_WINDOW,
    batch_size: int = BATCH_SIZE,
    seed: int = SEED,
    log_every: int = 50,
    split_name: str = "",
) -> dict[str, float]:
    """Preload `window_size` shards → run batches → discard → next window → clear."""
    if train:
        model.train()
        assert opt is not None
    else:
        model.eval()

    total_loss = 0.0
    n = 0
    correct = 0
    n_batches = 0
    tag = split_name or ("train" if train else "eval")

    for window, idxs in iter_shard_windows(
        dataset.entries,
        window_size=window_size,
        seed=seed,
        epoch=epoch,
        shuffle=shuffle,
    ):
        names = [cache.paths[si].name for si in window]
        print(f"  [{tag}] window {names}", flush=True)
        cache.preload(window)

        for start in range(0, len(idxs), batch_size):
            batch_idxs = idxs[start : start + batch_size]
            xb, y = torch.utils.data.dataloader.default_collate(
                [dataset[i] for i in batch_idxs]
            )
            xb = xb.to(DEVICE)
            y = y.to(DEVICE)

            if train:
                opt.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = policy_loss(logits, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                opt.step()
            else:
                with torch.no_grad():
                    logits = model(xb)
                    loss = policy_loss(logits, y)

            bs = xb.size(0)
            total_loss += float(loss.item()) * bs
            n += bs
            with torch.no_grad():
                pred = logits.view(bs, -1).argmax(dim=-1)
                correct += int((pred == y).sum())

            n_batches += 1
            if train and (n_batches == 1 or n_batches % log_every == 0):
                print(f"  epoch {epoch:02d}  batch {n_batches}  samples={n:,}", flush=True)

        cache.clear()

    return {
        "loss": total_loss / max(n, 1),
        "acc": correct / max(n, 1),
    }


def fmt_metrics(prefix: str, m: dict[str, float]) -> str:
    return f"{prefix}_loss={m['loss']:.4f}  {prefix}_acc={m['acc']:.3f}"

## 6. Train single model + save

In [24]:
set_seed(SEED)

# Shared cache: at most SHARD_WINDOW shards resident (~16GB). Never train+val together.
shard_cache = ShardCache(shard_paths, maxsize=SHARD_WINDOW)
train_ds = MultiShardBCDataset(shard_paths, sample_index, train_mask, cache=shard_cache)
val_ds = MultiShardBCDataset(shard_paths, sample_index, val_mask, cache=shard_cache)
test_ds = MultiShardBCDataset(shard_paths, sample_index, test_mask, cache=shard_cache)
print(f"shard window={SHARD_WINDOW}  (~{SHARD_WINDOW * 4}GB peak)  load→train→discard→repeat")

model = GeneralsBCNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

best_val = float("inf")
best_state = None
stale = 0
history = []

print(
    f"training pyr_cnn  |  train={len(train_ds):,}  val={len(val_ds):,}  "
    f"test={len(test_ds):,}  epochs<={MAX_EPOCHS}  patience={PATIENCE}"
)

for epoch in range(1, MAX_EPOCHS + 1):
    # Train: windows of 4 shards; each window cleared before the next.
    train_metrics = run_shard_windows(
        model,
        train_ds,
        shard_cache,
        train=True,
        opt=opt,
        epoch=epoch,
        shuffle=True,
        split_name="train",
    )

    # Val only after the full train epoch; cache already empty from last train window.
    val_metrics = run_shard_windows(
        model,
        val_ds,
        shard_cache,
        train=False,
        epoch=epoch,
        shuffle=False,
        split_name="val",
    )
    shard_cache.clear()

    history.append(
        {
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
    )
    print(
        f"epoch {epoch:02d}  {fmt_metrics('train', train_metrics)}  |  "
        f"{fmt_metrics('val', val_metrics)}"
    )

    if val_metrics["loss"] < best_val - 1e-4:
        best_val = val_metrics["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f"early stopping at epoch {epoch} (best val_loss={best_val:.4f})")
            break

if best_state is not None:
    model.load_state_dict(best_state)

# Test once at the end with a clean cache (no train/val shards left).
shard_cache.clear()
test_metrics = run_shard_windows(
    model,
    test_ds,
    shard_cache,
    train=False,
    epoch=0,
    shuffle=False,
    split_name="test",
)
shard_cache.clear()
print(f"\nTEST  {fmt_metrics('test', test_metrics)}")
print(f"best val_loss={best_val:.4f}")

CKPT_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = CKPT_DIR / CKPT_NAME
torch.save(
    {
        "model": best_state if best_state is not None else model.state_dict(),
        "test": test_metrics,
        "best_val_loss": best_val,
        "history": history,
        "config": {
            "seed": SEED,
            "train_frac": TRAIN_FRAC,
            "val_frac": VAL_FRAC,
            "test_frac": TEST_FRAC,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "lr_finetune": LR_FINETUNE,
            "grad_clip_norm": GRAD_CLIP_NORM,
            "nenvs": NENVS,
            "nsteps": NSTEPS,
            "truncation": TRUNCATION,
            "gamma": GAMMA,
            "grid_size": GRID_SIZE,
            "n_policy": N_POLICY,
            "n_train_games": len(split_ids["train"]),
            "n_val_games": len(split_ids["val"]),
            "n_test_games": len(split_ids["test"]),
        },
        "split_game_ids": {
            "train": sorted(split_ids["train"]),
            "val": sorted(split_ids["val"]),
            "test": sorted(split_ids["test"]),
        },
    },
    ckpt_path,
)
print(f"saved {ckpt_path.resolve()}")


shard window=4  (~16GB peak)  load→train→discard→repeat
training pyr_cnn  |  train=985,336  val=120,544  test=103,016  epochs<=20  patience=5
  [train] window ['bc_shard_0017_n32_t1001.npz', 'bc_shard_0064_n32_t1001.npz', 'bc_shard_0039_n32_t1001.npz', 'bc_shard_0015_n32_t1001.npz']
  [shard] load bc_shard_0017_n32_t1001.npz ...
  [shard] ready bc_shard_0017_n32_t1001.npz  obs=(147600, 14, 23, 23)  ~4.37GB  cached=1/4
  [shard] load bc_shard_0064_n32_t1001.npz ...
  [shard] ready bc_shard_0064_n32_t1001.npz  obs=(104520, 14, 23, 23)  ~3.10GB  cached=2/4
  [shard] load bc_shard_0039_n32_t1001.npz ...
  [shard] ready bc_shard_0039_n32_t1001.npz  obs=(162400, 14, 23, 23)  ~4.81GB  cached=3/4
  [shard] load bc_shard_0015_n32_t1001.npz ...
  [shard] ready bc_shard_0015_n32_t1001.npz  obs=(111360, 14, 23, 23)  ~3.30GB  cached=4/4
  epoch 01  batch 1  samples=600
  epoch 01  batch 50  samples=30,000


KeyboardInterrupt: 